In [1]:
# | default_exp models

# Models
> SQlModel Models of the Website.

In [2]:
# | export
from sqlmodel import Field, SQLModel, UniqueConstraint, Session, select, Column
from sqlalchemy import JSON
from datetime import datetime
from seootter.sqlite_db import get_session
import re
from pydantic import field_validator
from pathlib import Path

In [3]:
# | exporti
class Website(SQLModel, table=True):
    "Represents a website being tracked for SEO analysis."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    url: str = Field(unique=True)  # Unique website URL
    name: str  # Display name of the website
    desc: str | None = None  # Optional description
    lang: str = "en"  # Language code (e.g. 'en', 'ar')
    created_at: datetime = Field(default_factory=datetime.now)
    site_type: str
    content_dir: str = ""

    @field_validator("url")
    @classmethod
    def validate_url(cls, v: str) -> str:
        from urllip.parse import urlparse

        parsed = urlparse(v)
        if not all([parsed.scheme, parsed.netloc]):
            raise ValueError(f"Invalid URL: {v}")
        return v

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not v or len(v.strip()) < 1:
            raise ValueError(f"Invalid Name: {v}")
        return v

    @field_validator("lang")
    @classmethod
    def validate_lang(cls, v: str) -> str:
        if not re.match(r"^[a-z]{2}(-[A-Z]{2})?$", v):
            raise ValueError(f"Invalid Language Code: {v}")
        return v

In [4]:
# | hide
#| eval: false
s = get_session()
with s:
    site = Website(
        url="https://kareemai.com",
        name="kareemai",
        lang="ar",
        desc="Kareem Personal Blog",
    )
    s.add(site)
    s.commit()
    print(f"Added: {site.name}")

AttributeError: '_GeneratorContextManager' object has no attribute 'add'

In [5]:
# | export
def get_all_websites(session: Session  # Active database session
                     ) -> list[Website]:
    "Return all websites from the database."
    return session.exec(select(Website)).all()

In [6]:
# |hide
#| eval: false
get_all_websites(s)


AttributeError: '_GeneratorContextManager' object has no attribute 'exec'

In [7]:
# | export
def delete_website(session: Session,  # Active database session
                   website_id: int  # ID of the website to delete
                   ) -> None:
    "Delete a website by ID, raises ValueError if not found."
    website = session.get(Website, website_id)
    if website is None:
        raise ValueError(f"Website with id {website_id} not found")
    session.delete(website)
    session.commit()

In [8]:
# | exporti
class TrackedKeyword(SQLModel, table=True):
    "A keyword being tracked for SERP position monitoring."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    website_id: int = Field(foreign_key="website.id")
    keyword: str
    added_at: datetime = Field(default_factory=datetime.now)


In [9]:
# | exporti
class URLMapping(SQLModel, table=True):
    "Maps a public URL to its local file path for a website."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    website_id: int = Field(foreign_key="website.id")
    url: str
    file_path: str = ""
    synced_at: datetime = Field(default_factory=datetime.now)


In [10]:
# | export
def get_tracked_keywords(session: Session,  # Active database session
                          website_id: int  # Parent website ID
                         ) -> list[TrackedKeyword]:
    "Get all tracked keywords for a website."
    return session.exec(select(TrackedKeyword).where(TrackedKeyword.website_id == website_id)).all()


def add_tracked_keyword(session: Session,  # Active database session
                         website_id: int,  # Parent website ID
                         keyword: str  # Keyword to track
                        ) -> TrackedKeyword:
    "Add a keyword for SERP monitoring."
    kw = TrackedKeyword(website_id=website_id, keyword=keyword)
    session.add(kw)
    session.commit()
    session.refresh(kw)
    return kw


def delete_tracked_keyword(session: Session,  # Active database session
                            keyword_id: int  # Keyword ID to remove
                           ) -> None:
    "Delete a tracked keyword by ID."
    kw = session.get(TrackedKeyword, keyword_id)
    if kw:
        session.delete(kw)
        session.commit()


In [11]:
# | export
def get_url_mapping(session: Session,  # Active database session
                     website_id: int  # Parent website ID
                    ) -> list[URLMapping]:
    "Get all URL→file mappings for a website."
    return session.exec(select(URLMapping).where(URLMapping.website_id == website_id)).all()


def sync_url_mapping(session: Session,  # Active database session
                      website_id: int,  # Parent website ID
                      mapping: dict[str, str]  # URL → file_path mapping
                     ) -> None:
    "Sync URL→file mappings, preserving existing entries and adding new ones."
    existing = {m.url: m for m in get_url_mapping(session, website_id)}
    for url, file_path in mapping.items():
        if url in existing:
            m = existing[url]
            if m.file_path != file_path:
                m.file_path = file_path
                session.add(m)
        else:
            session.add(URLMapping(website_id=website_id, url=url, file_path=file_path))
    session.commit()


In [12]:
# | hide
# delete_website(db.get_session(), website_id=1)

In [13]:
# | export

def add_or_update_website(
        session: Session, url: str, name: str, site_type: str,
        desc: str = "", lang: str = "en", content_dir: str = "",
        website_id: int | None = None
) -> Website:
    """Add a new website or update an exisiting one."""
    if website_id:
        website = session.get(Website, website_id)
        if website is None:
            raise ValueError(f"Website with id {website_id} not found")
    else:
        website = session.exec(select(Website).where(Website.url == url)).first()
    if website:
        website.name = name
        website.site_type = site_type
        website.desc = desc or website.desc
        website.lang = lang
        website.content_dir = content_dir
    else:
        website = Website(
            url=url, name=name, site_type=site_type,
            desc=desc, lang=lang, content_dir=content_dir
        )
        session.add(website)
    session.commit()
    session.refresh(website)
    return website


In [14]:
#| export
from rich.table import Table
from rich import print as rprint


def print_websites(websites: list[Website]) -> None:
    "Print websites as a rich table."
    if not websites:
        rprint("[yellow]No websites found. Add one with: seo-otter-add-website[/yellow]")
        return
    table = Table(title="🌐 Websites")
    table.add_column("ID", justify="right", style="dim")
    table.add_column("Name", style="cyan")
    table.add_column("URL")
    table.add_column("Type", style="green")
    table.add_column("Lang", justify="center")
    for w in websites:
        table.add_row(str(w.id), w.name, w.url, w.site_type, w.lang)
    rprint(table)

In [15]:
# | exporti
class GSCAnalytics(SQLModel, table=True):
    "Google Search Console analytics data for a site, query, page, and device combination."
    __table_args__ = (
        UniqueConstraint("site_url", "date", "query", "page", "country", "device"),
        {"extend_existing": True},
    )
    id: int | None = Field(default=None, primary_key=True)
    site_url: str  # The verified GSC property URL
    date: str  # Date of the analytics record (YYYY-MM-DD)
    query: str | None = None  # Search query keyword
    page: str | None = None  # Page URL
    country: str | None = None  # Country code (e.g. 'usa')
    device: str | None = None  # Device type (e.g. 'mobile')
    clicks: int = 0
    impressions: int = 0
    ctr: float = 0.0
    position: float = 0.0
    created_at: datetime = Field(default_factory=datetime.now)



In [ ]:
#| export
class GSCPropertyTotals(SQLModel, table=True):
    "Google Search Console property-level analytics totals."
    __table_args__ = (
        UniqueConstraint("site_url", "date"),
        {"extend_existing": True},
    )
    id: int | None = Field(default=None, primary_key=True)
    site_url: str
    date: str
    clicks: int = 0
    impressions: int = 0
    ctr: float = 0.0
    position: float = 0.0
    created_at: datetime = Field(default_factory=datetime.now)


In [16]:
# | exporti
class IndexStatus(SQLModel, table=True):
    "Google Search Console indexing status for a specific page."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    site_url: str  # The verified GSC property URL
    page_url: str  # The specific page being checked
    verdict: str  # Overall indexing verdict (e.g. 'PASS', 'FAIL')
    coverage_state: str | None = None  # Coverage state from GSC
    last_crawl_time: str | None = None  # Last time Google crawled the page
    indexing_state: str | None = None  # Whether the page is indexed
    robots_txt_state: str | None = None  # Robots.txt blocking status
    checked_at: datetime = Field(default_factory=datetime.now)



## Wuilt Models

In [17]:
#| exporti
class WuiltStore(SQLModel, table=True):
    "A Wuilt e-commerce store registered for SEO optimization."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    store_id: str = Field(unique=True)  # Wuilt store ID ("Store_...")
    name: str  # Display name
    api_key: str  # Wuilt GraphQL API key
    locale: str = "ar"  # Store locale ("ar", "en")
    store_domain: str = ""  # Storefront domain (for GSC correlation)
    website_id: int | None = Field(default=None, foreign_key="website.id")  # Optional link to Website
    created_at: datetime = Field(default_factory=datetime.now)


class WuiltProduct(SQLModel, table=True):
    "A product synced from a Wuilt store, with its SEO fields."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    wuilt_store_id: int = Field(foreign_key="wuiltstore.id")
    wuilt_product_id: str  # Wuilt's product ID
    title: str
    handle: str
    seo_title: str | None = None
    seo_description: str | None = None
    description_html: str | None = None
    short_description: str | None = None
    price: float | None = None
    image_urls: str | None = Field(default=None, sa_column=Column(JSON))  # list[str]
    images_alt: str | None = Field(default=None, sa_column=Column(JSON))  # list[str]
    raw_data: str | None = Field(default=None, sa_column=Column(JSON))  # full GraphQL snapshot
    synced_at: datetime = Field(default_factory=datetime.now)
    last_optimized_at: datetime | None = None
    optimized_seo_title: str | None = None
    optimized_seo_description: str | None = None
    optimized_description_html: str | None = None


In [18]:
#| export
def get_all_wuilt_stores(session: Session) -> list[WuiltStore]:
    "Return all registered Wuilt stores."
    return session.exec(select(WuiltStore)).all()


def add_or_update_wuilt_store(
    session: Session,
    store_id: str,
    name: str,
    api_key: str,
    locale: str = "ar",
    store_domain: str = "",
    website_id: int | None = None,
    store_pk: int | None = None,
) -> WuiltStore:
    """Add or update a Wuilt store registration."""
    if store_pk:
        store = session.get(WuiltStore, store_pk)
        if store is None:
            raise ValueError(f"WuiltStore with id {store_pk} not found")
    else:
        store = session.exec(select(WuiltStore).where(WuiltStore.store_id == store_id)).first()
    if store:
        store.name = name
        store.api_key = api_key
        store.locale = locale
        store.store_domain = store_domain
        store.website_id = website_id
    else:
        store = WuiltStore(
            store_id=store_id, name=name, api_key=api_key,
            locale=locale, store_domain=store_domain, website_id=website_id,
        )
        session.add(store)
    session.commit()
    session.refresh(store)
    return store


def delete_wuilt_store(session: Session, store_pk: int) -> None:
    "Delete a Wuilt store by ID."
    store = session.get(WuiltStore, store_pk)
    if store is None:
        raise ValueError(f"WuiltStore with id {store_pk} not found")
    session.delete(store)
    session.commit()


def get_wuilt_products(session: Session, wuilt_store_id: int) -> list[WuiltProduct]:
    "Get all synced products for a Wuilt store."
    return session.exec(
        select(WuiltProduct).where(WuiltProduct.wuilt_store_id == wuilt_store_id)
    ).all()


def upsert_wuilt_product(
    session: Session,
    wuilt_store_id: int,
    wuilt_product_id: str,
    title: str,
    handle: str,
    seo_title: str | None = None,
    seo_description: str | None = None,
    description_html: str | None = None,
    short_description: str | None = None,
    price: float | None = None,
    image_urls: list[str] | None = None,
    images_alt: list[str] | None = None,
    raw_data: dict | None = None,
) -> WuiltProduct:
    "Insert or update a Wuilt product in local DB."
    product = session.exec(
        select(WuiltProduct).where(
            WuiltProduct.wuilt_store_id == wuilt_store_id,
            WuiltProduct.wuilt_product_id == wuilt_product_id,
        )
    ).first()
    if product:
        product.title = title
        product.handle = handle
        product.seo_title = seo_title
        product.seo_description = seo_description
        product.description_html = description_html
        product.short_description = short_description
        product.price = price
        product.image_urls = image_urls
        product.images_alt = images_alt
        product.raw_data = raw_data
        product.synced_at = datetime.now()
    else:
        product = WuiltProduct(
            wuilt_store_id=wuilt_store_id,
            wuilt_product_id=wuilt_product_id,
            title=title, handle=handle,
            seo_title=seo_title, seo_description=seo_description,
            description_html=description_html, short_description=short_description,
            price=price, image_urls=image_urls, images_alt=images_alt, raw_data=raw_data,
        )
        session.add(product)
    session.commit()
    session.refresh(product)
    return product


In [ ]:
#| export

class WuiltPage(SQLModel, table=True):
    "A store page synced from a Wuilt store, with its SEO fields."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    wuilt_store_id: int = Field(foreign_key="wuiltstore.id")
    wuilt_page_id: str  # Wuilt's page ID
    name: str
    handle: str
    page_type: str = "CUSTOM"
    status: str = "DRAFT"
    locale: str = "ar"
    seo_title: str | None = None
    seo_description: str | None = None
    raw_data: str | None = Field(default=None, sa_column=Column(JSON))
    synced_at: datetime = Field(default_factory=datetime.now)
    last_optimized_at: datetime | None = None
    optimized_seo_title: str | None = None
    optimized_seo_description: str | None = None


In [19]:
#| export
def print_wuilt_stores(stores: list[WuiltStore]) -> None:
    "Print Wuilt stores as a rich table."
    if not stores:
        rprint("[yellow]No Wuilt stores registered.[/yellow]")
        return
    table = Table(title="🏪 Wuilt Stores")
    table.add_column("ID", justify="right", style="dim")
    table.add_column("Name", style="cyan")
    table.add_column("Store ID")
    table.add_column("Locale", justify="center")
    table.add_column("Domain")
    for s in stores:
        table.add_row(str(s.id), s.name, s.store_id, s.locale, s.store_domain or "-")
    rprint(table)


def print_wuilt_products(products: list[WuiltProduct]) -> None:
    "Print Wuilt products as a rich table."
    if not products:
        rprint("[yellow]No products synced yet.[/yellow]")
        return
    table = Table(title="📦 Wuilt Products")
    table.add_column("ID", justify="right", style="dim")
    table.add_column("Title", style="cyan")
    table.add_column("Handle")
    table.add_column("Price", justify="right")
    table.add_column("SEO Title")
    table.add_column("Optimized")
    for p in products:
        opt = "✅" if p.last_optimized_at else "—"
        table.add_row(
            str(p.id), p.title, p.handle,
            f"{p.price:.0f}" if p.price else "-",
            (p.seo_title or "")[:40],
            opt,
        )
    rprint(table)


In [ ]:
#| hide

from dotenv import load_dotenv

load_dotenv()
import os

WUILT_STORE_ID = os.getenv("WUILT_STORE_ID")
WUILT_API_KEY = os.getenv("WUILT_API_KEY")

with get_session("sqlite:///test_wuilt.db") as s:
    store = add_or_update_wuilt_store(
        s,
        store_id=WUILT_STORE_ID,
        name="HelixGain",
        api_key=WUILT_API_KEY,
        locale="ar",
        store_domain="https://cm9pmn9ow062e01h337yh7wev.wuiltstore.com/ar",
    )
    print(f"✅ Store saved: {store.name} (pk={store.id})")

    p = upsert_wuilt_product(
        s,
        wuilt_store_id=store.id,
        wuilt_product_id="prod_test_001",
        title="Test",
        handle="test",
        price=100.0,
        seo_title="Test | Buy Now",
        seo_description="Best test product",
        image_urls=["https://assets.wuiltstore.com/hash.webp"],
        images_alt=["Test image"],
    )
    print(f"✅ Product saved: {p.title}")

    print_wuilt_stores(get_all_wuilt_stores(s))
    print_wuilt_products(get_wuilt_products(s, store.id))

    delete_wuilt_store(s, store.id)

✅ Store saved: HelixGain (pk=1)
✅ Product saved: Test


                                                  🏪 Wuilt Stores                                                  
┏━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID ┃ Name      ┃ Store ID                        ┃ Locale ┃ Domain                                              ┃
┡━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ HelixGain │ Store_cm9pmn9ow062d01h3ep654q8e │   ar   │ https://cm9pmn9ow062e01h337yh7wev.wuiltstore.com/ar │
└────┴───────────┴─────────────────────────────────┴────────┴─────────────────────────────────────────────────────┘

                               📦 Wuilt Products                               
┏━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ ID ┃ Title           ┃ Handle          ┃ Price ┃ SEO Title      ┃ Optimized ┃
┡━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│  1 │ Updated Product │ updated-product │   349 │                │ —         │
│  2 │ Test            │ test            │   100 │ Test | Buy Now │ —         │
└────┴─────────────────┴─────────────────┴───────┴────────────────┴───────────┘

In [ ]:
#| export
def get_wuilt_pages(session: Session, wuilt_store_id: int) -> list[WuiltPage]:
    "Get all synced pages for a Wuilt store."
    return session.exec(
        select(WuiltPage).where(WuiltPage.wuilt_store_id == wuilt_store_id)
    ).all()


def upsert_wuilt_page(
    session: Session,
    wuilt_store_id: int,
    wuilt_page_id: str,
    name: str,
    handle: str,
    page_type: str = "CUSTOM",
    status: str = "DRAFT",
    locale: str = "ar",
    seo_title: str | None = None,
    seo_description: str | None = None,
    raw_data: dict | None = None,
) -> WuiltPage:
    "Insert or update a Wuilt page in local DB."
    page = session.exec(
        select(WuiltPage).where(
            WuiltPage.wuilt_store_id == wuilt_store_id,
            WuiltPage.wuilt_page_id == wuilt_page_id,
        )
    ).first()
    if page:
        page.name = name
        page.handle = handle
        page.page_type = page_type
        page.status = status
        page.locale = locale
        page.seo_title = seo_title
        page.seo_description = seo_description
        page.raw_data = raw_data
        page.synced_at = datetime.now()
    else:
        page = WuiltPage(
            wuilt_store_id=wuilt_store_id,
            wuilt_page_id=wuilt_page_id,
            name=name, handle=handle,
            page_type=page_type, status=status, locale=locale,
            seo_title=seo_title, seo_description=seo_description,
            raw_data=raw_data,
        )
        session.add(page)
    session.commit()
    session.refresh(page)
    return page


def print_wuilt_pages(pages: list[WuiltPage]) -> None:
    "Print Wuilt pages as a rich table."
    if not pages:
        rprint("[yellow]No pages synced yet.[/yellow]")
        return
    table = Table(title="📄 Wuilt Pages")
    table.add_column("ID", justify="right", style="dim")
    table.add_column("Name", style="cyan")
    table.add_column("Handle")
    table.add_column("Type")
    table.add_column("Status")
    table.add_column("SEO Title")
    table.add_column("Optimized")
    for p in pages:
        opt = "✅" if p.last_optimized_at else "—"
        table.add_row(
            str(p.id), p.name, p.handle,
            p.page_type, p.status,
            (p.seo_title or "")[:40],
            opt,
        )
    rprint(table)
